# 🤖 AskSQL: Anime Database Text-to-SQL System

Welcome to the **AskSQL** demo. This notebook serves as the main interface for the project, demonstrating the end-to-step pipeline: from data loading and training to inference.


<a id='setup'></a>
## 1. Setup Environment
Initializing the project environment, loading the trained model, and connecting to the database.

In [1]:
import sys
import os
import pandas as pd
from IPython.display import display, HTML, Markdown

# Add src to path to import modules
sys.path.append(os.path.abspath(''))

from src.inference import SQLGenerator
from src.database import execute_query, init_database

# Ensure Database Exists
if not os.path.exists('project_db.sqlite'):
    display(Markdown("*Initializing database from CSV...*"))
    init_database()

# Load Model
try:
    display(Markdown("*Loading T5 Model...*"))
    gen = SQLGenerator(model_path='models/avgs_t5_wikisql')
    display(Markdown("**✅ System Ready!**"))
except Exception as e:
    display(Markdown(f"**⚠️ Error loading model:** {e}"))

*Loading T5 Model...*

Loading model from models/avgs_t5_wikisql...
Successfully loaded model from models/avgs_t5_wikisql


**✅ System Ready!**

<a id='data'></a>
## 2.  Data Overview
Let's look at the raw data used for this project.

In [2]:
csv_path = 'data/animenew.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    display(Markdown(f"**Loaded `{csv_path}` with {len(df)} rows.**"))
    display(df.head())
else:
    display(Markdown("⚠️ Data file not found."))

**Loaded `data/animenew.csv` with 14578 rows.**

,title,mediaType,eps,duration,ongoing,startYr,finishYr,sznOfRelease,description,studios,tags,contentWarn,watched,watching,wantWatch,dropped,rating,votes
0,Fullmetal Alchemist: Brotherhood,TV,64.0,NaN,False,2009.0,2010.0,Spring,The foundation of alchemy is based on the law ...,['Bones'],"['Action', 'Adventure', 'Drama', 'Fantasy', 'M...","['Animal Abuse', 'Mature Themes', 'Violence', ...",103707.0,14351,25810,2656,4.702,86547.0
1,your name.,Movie,1.0,107.0,False,2016.0,2016.0,NaN,Mitsuha and Taki are two total strangers livin...,['CoMix Wave Films'],"['Drama', 'Romance', 'Body Swapping', 'Gender ...",[],58831.0,1453,21733,124,4.663,43960.0
2,A Silent Voice,Movie,1.0,130.0,False,2016.0,2016.0,NaN,"After transferring into a new school, a deaf g...",['Kyoto Animation'],"['Drama', 'Shounen', 'Disability', 'Melancholy...","['Bullying', 'Mature Themes', 'Suicide']",45892.0,946,17148,132,4.661,33752.0
3,Haikyuu!! Karasuno High School vs Shiratorizaw...,TV,10.0,NaN,False,2016.0,2016.0,Fall,"Picking up where the second season ended, the ...",['Production I.G'],"['Shounen', 'Sports', 'Animeism', 'School Club...",[],25134.0,2183,8082,167,4.660,17422.0
4,Attack on Titan 3rd Season: Part II,TV,10.0,NaN,False,2019.0,2019.0,Spring,The battle to retake Wall Maria begins now! Wi...,['Wit Studio'],"['Action', 'Fantasy', 'Horror', 'Shounen', 'Da...","['Cannibalism', 'Explicit Violence']",21308.0,3217,7864,174,4.650,15789.0


### 2.1 Database Schema
The data is loaded into a SQLite table named `anime`. Here are the available columns you can query:

| Column | Type | Description |
| :--- | :--- | :--- |
| `title` | TEXT | The name of the anime. |
| `mediaType` | TEXT | Format (e.g., TV, Movie, OVA). |
| `eps` | REAL | Number of episodes. |
| `duration` | REAL | Duration in minutes (per episode). |
| `ongoing` | INTEGER | Whether it is currently airing (1/0). |
| `startYr` | REAL | Year the anime started. |
| `finishYr` | REAL | Year the anime finished. |
| `sznOfRelease` | TEXT | Season of release (e.g., Spring, Fall). |
| `description` | TEXT | Plot summary or synopsis. |
| `studios` | TEXT | Production studio(s). |
| `tags` | TEXT | Genres and tags (e.g., Action, Comedy). |
| `contentWarn` | TEXT | Content warnings. |
| `watched` | REAL | Number of users who watched it. |
| `watching` | REAL | Number of users currently watching. |
| `wantWatch` | REAL | Number of users who want to watch. |
| `dropped` | REAL | Number of users who dropped it. |
| `rating` | REAL | Average user rating (0-5). |
| `votes` | REAL | Number of votes for the rating. |

<a id='train'></a>
## 3.Training Process
The following cells demonstrate how to generate synthetic training data and train the model. 

> **Note:** Training can take some time. The cells below execute the scripts in the `scripts/` directory.

### 3.1 Generate Synthetic Data
Creates `train_processed.csv` and `dev_processed.csv` from the raw data.

In [3]:
# !python scripts/generate_synthetic.py

### 3.2 Train Model
Runs the training script using the generated data. This applies LoRA fine-tuning to the T5 model.

In [4]:
# !python scripts/train_model.py

<a id='inference'></a>
## 4. Interactive Inference
This section provides an interactive loop where you can continuously ask questions.  
**Type 'exit' or 'quit' to stop the loop.**

In [5]:
def ask_sql(question, table_id="anime"):
    # Standard Schema Definition
    headers = ["title", "mediaType", "eps", "duration", "ongoing", "startYr", "finishYr", "sznOfRelease", "description", "studios", "tags", "contentWarn", "watched", "watching", "wantWatch", "dropped", "rating", "votes"]
    types = ["TEXT", "TEXT", "REAL", "REAL", "INTEGER", "REAL", "REAL", "TEXT", "TEXT", "TEXT", "TEXT", "TEXT", "REAL", "REAL", "REAL", "REAL", "REAL", "REAL"]
    
    # Display Question
    display(HTML(f"<div style='background-color: #f0f2f6; padding: 10px; border-radius: 5px; border-left: 5px solid #3498db; margin-bottom: 10px;'><h3>❓ {question}</h3></div>"))
    
    # 1. Generate SQL
    try:
        sql = gen.generate_sql(question, table_id, headers, types)
        display(HTML(f"<div style='background-color: #2c3e50; color: #ecf0f1; padding: 10px; border-radius: 5px; font-family: monospace; margin-bottom: 10px;'>&gt; {sql}</div>"))
        
        # 2. Execute SQL
        df, err = execute_query(sql)
        
        if err:
            display(HTML(f"<div style='color: #c0392b; background-color: #fadbd8; padding: 10px; border-radius: 5px;'>⚠️ <b>Execution Error:</b> {err}</div>"))
        else:
            if df.empty:
                 display(HTML("<i>No results found.</i>"))
            else:
                display(HTML(f"<b>✅ Found {len(df)} results:</b>"))
                display(df.style.set_table_styles(
                    [{'selector': 'th', 'props': [('background-color', '#34495e'), ('color', 'white'), ('font-family', 'sans-serif')]},
                     {'selector': 'td', 'props': [('padding', '8px'), ('border-bottom', '1px solid #ddd')]}]
                ).hide(axis='index'))
                
    except Exception as e:
         display(HTML(f"<div style='color: red;'>System Error: {e}</div>"))
    
    display(HTML("<hr style='border-top: 1px dashed #bbb;'>"))

# Interactive Loop
print("Starting interactive mode. Type 'exit' to stop.")
while True:
    try:
        user_input = input("Enter your question: ")
        if user_input.lower() in ['exit', 'quit']:
            print("Exiting interactive mode.")
            break
        if user_input.strip():
            ask_sql(user_input)
    except KeyboardInterrupt:
        print("\nStopped by user.")
        break

Starting interactive mode. Type 'exit' to stop.


Enter your question:  list top 5 animes


DEBUG INPUT: tables: CREATE TABLE anime (title TEXT, mediaType TEXT, eps REAL, duration REAL, ongoing INTEGER, startYr REAL, finishYr REAL, sznOfRelease TEXT, description TEXT, studios TEXT, tags TEXT, contentWarn TEXT, watched REAL, watching REAL, wantWatch REAL, dropped REAL, rating REAL, votes REAL) query for: list top 5 animes
DEBUG RAW OUTPUT: SELECT title FROM anime ORDER BY ratings DESC LIMIT 5


title
Fullmetal Alchemist: Brotherhood
your name.
A Silent Voice
Haikyuu!! Karasuno High School vs Shiratorizawa Academy
Attack on Titan 3rd Season: Part II


Enter your question:  exit


Exiting interactive mode.


<a id='eval'></a>
## 5.Evaluation Examples
test runs to verify the system performance.

### Test Run 1: Simple Retrieval
Checking if the model can retrieve text attributes.

In [6]:
ask_sql("What is the story of One Piece?")

description
"Twenty years ago, the notorious pirate, Gold Lion Shiki, managed to achieve the impossible: he became the first person to escape from the maximum-security marine prison, Impel Down. Having spent the last two decades in hiding, Shiki has now returned, and he has a series of nefarious plans in mind. One day, after a chance encounter with Luffy and the gang, Shiki takes a liking to Nami’s talents as a navigator and decides to claim her for his own, scattering the rest of the crew across his various floating islands in the sky. Now, Luffy and the Straw Hats must fight their way past Skiki’s army of modified animals in order to save their comrade and stop the Gold Lion’s plot to destroy East Blue and gain control over the world."
"The world’s boldest buccaneers set sail for the great Pirate Festival, where the Straw Hats join a mad-dash race to find Gol D. Roger’s treasure. There’s just one little problem. An old member of Roger’s crew has a sinister score to settle."
"Said to be comparable to the Ancient Weapons of old, the Marines' trump card, the ""Dyna Stones,"" have suddenly been stolen by a group of renegade vigilantes. The terrifyingly powerful man responsible, former Marine Admiral ""Z"", now stands in the path of Luffy and his Straw Hat Pirates. Can the Straw Hats defeat ""Z"" and his crew, or will the New World meet its end at the hands of this mad man?"
"Long ago the infamous Gol D. Roger was the strongest and most powerful pirate on the seas. As he was about to be executed he revealed that he hid all of his wealth, including the legendary treasure known as One Piece, on an island at the end of the Grand Line - a treacherous and truly unpredictable sea. Monkey D. Luffy is a spirited, energetic and somewhat dim-witted young man with a very big dream: to find One Piece and become the Pirate King! However Luffy is no ordinary boy, as when he was younger he ate one of the Devil's Fruits and gained its power to become a Rubber Man. Now in this grand age of pirates Luffy sets out to gather a crew and sail to the most dangerous sea in the world so that he can fulfill his dream... and maybe even his appetite!"
"The glittering Gran Tesoro, a city of entertainment beyond the laws of the government, is a sanctuary for the world’s most infamous pirates, Marines, and filthy rich millionaires. Drawn by dreams of hitting the jackpot, Captain Luffy and his crew sail straight for the gold. But behind the gilded curtains lies a powerful king whose deep pockets and deeper ambitions spell disaster for the Straw Hats and the New World alike."
"On a test-run of the Mini Merry, Usopp tells Brook the story of the Going Merry, a ship that served the crew well. However, to the Straw Hats, the Going Merry was not just a ship: It was a priceless, irreplaceable friend."
"After Nami steals the Going Merry, Roronoa Zoro, Usopp, and later Monkey D. Luffy and Sanji set after her and wind up in Cocoyasi Village, Nami's hometown ruled by the tyrannical fishman Arlong. It is here that Nami's past and true motives come to light."
"While training under Rayleigh, Luffy must learn to harness the power of Haki in order to save the beautiful Boa Hancock and her sisters from a deadly escaped prisoner!"
"Many years before his showdown with Luffy, the 'Gold Lion' Shiki came face to face with the most legendary pirate of them all: Gol D. Roger. On one fateful day the two fearsome captains clash in a fight known as ""The Great Battle of Edd War"" and Shiki and his pirates are forced to flee the battle, regrouping later on. However, when Gol D. Roger is captured three years later, Shiki refuses to accept it and decides to go up against the navy to stop his execution and claim the pirate king’s life himself..."
"The Straw Hats are about to set sail on the Grand Line! Monkey D. Luffy, a rubber man; Roronoa Zoro aka “Pirate Hunter;” Usopp, a sniper; Sanji, a seafaring cook; and “Cat Burglar” Nami make their vows before their next journey across the great oce

### Test Run 2: Filtering with Logic
Checking numerical comparison and conditions.

In [7]:
ask_sql("List top 5 movies with most watchour ")

DEBUG INPUT: tables: CREATE TABLE anime (title TEXT, mediaType TEXT, eps REAL, duration REAL, ongoing INTEGER, startYr REAL, finishYr REAL, sznOfRelease TEXT, description TEXT, studios TEXT, tags TEXT, contentWarn TEXT, watched REAL, watching REAL, wantWatch REAL, dropped REAL, rating REAL, votes REAL) query for: List top 5 movies with most watchour 
DEBUG RAW OUTPUT: SELECT title, MIN(watched) FROM anime GROUP BY title ORDER BY SUM(watched) DESC LIMIT 5


title,MIN(watched)
Death Note,161567.000000
Attack on Titan,121565.000000
Naruto,119474.000000
Sword Art Online,117831.000000
Spirited Away,115949.000000


### Test Run 3: Ranking/Top N
Checking sorting and limit functionality.

In [8]:
ask_sql("List top 5 rated anime")

title,rating
Fullmetal Alchemist: Brotherhood,4.702000
your name.,4.663000
A Silent Voice,4.661000
Haikyuu!! Karasuno High School vs Shiratorizawa Academy,4.660000
Attack on Titan 3rd Season: Part II,4.650000
